# The Most Beautiful Disulfide Bond in the World
Eric G. Suchanek, PhD 5/4/24

In this notebook I illustrate some of the features of proteusPy by analyzing the lowest energy disulfide bond in the RCSB protein structure database. If you are not familiar with `proteusPy` you can find the API at: https://suchanek.githubio.com/proteusPy.html


In [1]:
import pyvista as pv

import proteusPy
from proteusPy import Disulfide, DisulfideList, Load_PDB_SS

# pyvista setup for notebooks
pv.set_jupyter_backend("trame")
pv.set_plot_theme("default")


proteusPy.__version__

'0.99.34.dev3'

### Load the RCSB Disulfide Database
We load the database and get its properties as follows:

In [ ]:
PDB_SS = Load_PDB_SS(verbose=True)

proteusPy: INFO 2025-03-28 07:21:40,356 - proteusPy.DisulfideLoader.Load_PDB_SS - Reading disulfides from: /Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages/proteusPy/data/PDB_SS_ALL_LOADER.pkl...
INFO:proteusPy.DisulfideLoader:Reading disulfides from: /Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages/proteusPy/data/PDB_SS_ALL_LOADER.pkl...
proteusPy: INFO 2025-03-28 07:21:46,534 - proteusPy.DisulfideLoader.Load_PDB_SS - Done reading disulfides from: /Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages/proteusPy/data/PDB_SS_ALL_LOADER.pkl...
INFO:proteusPy.DisulfideLoader:Done reading disulfides from: /Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages/proteusPy/data/PDB_SS_ALL_LOADER.pkl...



    🌟 RCSB Disulfide Database Summary 🌟
       🕒 Constructed: 2025-03-27 22:48:19 🕒
PDB IDs Present:               35375
Disulfides Loaded:             158938
Average Resolution:            2.18 Å
Lowest Energy Disulfide:       2q7q_75D_140D
Highest Energy Disulfide:      6vxk_801B_806B
Cα Distance Cutoff:            6.71 Å
Sγ Distance Cutoff:            2.12 Å
Percentile Cutoff:             95.00 %
     ⚡ proteusPy Version: 0.99.34.dev3 ⚡


    🌟 RCSB Disulfide Database Summary 🌟
       🕒 Constructed: 2025-03-27 22:48:19 🕒
PDB IDs Present:               35375
Disulfides Loaded:             158938
Average Resolution:            2.18 Å
Lowest Energy Disulfide:       2q7q_75D_140D
Highest Energy Disulfide:      6vxk_801B_806B
Cα Distance Cutoff:            6.71 Å
Sγ Distance Cutoff:            2.12 Å
Percentile Cutoff:             95.00 %
     ⚡ proteusPy Version: 0.99.34.dev3 ⚡



We see from the statistics above that disulfide 2q7q_75D_140D has the lowest energy, so let's extract it from the database and display it. 

A few notes about the display window. You might need to click into the window to refresh it. Click drag to rotate the structures, mousewheel to zoom. The window titles display several parameters about the disulfide bonds including their approximate torsional energy, their Ca-Ca distance, and the *torsion length*. 

The latter parameter is formally, the Euclidean length of the sidechain dihedral angle when treated as a five-dimensional vector. This sounds all mathy and complicated, but in essence it gives a measure of how 'long' that five dimensional vector is. This is used by the package to compare individual structures and gauge their structural similarity.

In [3]:
ssmin, ssmax = PDB_SS.SSList.minmax_energy
ssmin_energy = ssmin.energy

best_ss = PDB_SS["2q7q_75D_140D"]
best_dihedrals = best_ss.dihedrals
# best_ss.pprint_all()
best_ss.display(style="cpk", light="Auto")

Widget(value='<iframe src="http://localhost:59052/index.html?ui=P_0x103fcfec0_0&reconnect=auto" class="pyvista…

And that, gentle reader, is it. *The most beautiful disulfide bond in the world*! Look at it. This is the lowest energy structure in the entire database. The sidechain dihdedral angles (Χ1-Χ5: -59.36°, -59.28°, -83.66°, -59.82° -59.91°), and the estimated energy, (0.49 kcal/mol). 

How does this compare to an analytical (modelled) minimum? We can use the ``minimize`` module from ``scipy`` to check. We know from chemistry that a reasonable guess for a low energy conformation would have the dihedral angles: (Χ1-Χ5: -60.00°, -60.00°, -90.00°, -60.00° -60.00°, 0.60 kcal/mol). Let's run this through scipy and compute a minimum energy conformation:


In [4]:
from proteusPy.Disulfide import disulfide_energy_function
from scipy.optimize import minimize

initial_guess = [
    -60.0,
    -60.0,
    -90.0,
    -60.0,
    -60.0,
]  # initial guess for chi1, chi2, chi3, chi4, chi5
result = minimize(disulfide_energy_function, initial_guess, method="Nelder-Mead")
minimum_energy = result.fun
minimum_conformation = result.x
print(
    f'Minimum energy: {minimum_energy:.3f} kcal/mol for conformation: {[f"{x:.3f}" for x in ssmin.dihedrals]}\n'
    f'Modeled minimum energy: {minimum_energy:.3f} kcal/mol for conformation: {[f"{x:.3f}" for x in minimum_conformation]}'
)

Minimum energy: 0.489 kcal/mol for conformation: ['-59.359', '-59.278', '-83.663', '-59.819', '-59.914']
Modeled minimum energy: 0.489 kcal/mol for conformation: ['-60.000', '-60.000', '-83.048', '-60.000', '-60.000']


So the computed minimum energy structure is *0.489* kcal/mol, estimated. The difference from the actual conformation is:

In [5]:
diff = minimum_energy - ssmin_energy
print(f"Modeled - actual energy difference is: {diff} kcal/mol")

Modeled - actual energy difference is: -0.0027979815677854347 kcal/mol


Given this very small difference we can safely say that the lowest energy disulfide in the database is at the lowest theoretical energy as well. What an amazing Disulfide bond! Let's build a model for the predicted lowest-energy conformation and compare it to the actual one found in the database. We do that by creating an empty disulfide and then using the `Disulfide.build_yourself` function.

In [ ]:
modelled_min = Disulfide("model", torsions=minimum_conformation)

<Disulfide model, Source: 1egs, Resolution: -1.0 Å
 
Proximal Coordinates:
   N: <Vector3D (-0.48, 1.37, -0.40)>
   Cα: <Vector3D (0.00, 0.00, 0.00)>
   C: <Vector3D (-0.52, -0.72, 1.25)>
   O: <Vector3D (0.00, 0.00, 0.00)>
   Cβ: <Vector3D (1.53, -0.00, -0.00)>
   Sγ: <Vector3D (2.25, 0.86, -1.48)>
   Cprev <Vector3D (0.00, 0.00, 0.00)>
   Nnext: <Vector3D (0.00, 0.00, 0.00)>
 Distal Coordinates:
   N: <Vector3D (4.20, -0.48, -3.42)>
   Cα: <Vector3D (4.14, -1.23, -3.45)>
   C: <Vector3D (5.04, -2.46, -3.55)>
   O: <Vector3D (0.00, 0.00, 0.00)>
   Cβ: <Vector3D (2.69, -1.66, -3.20)>
   Sγ: <Vector3D (1.52, -0.21, -3.07)>
   Cprev <Vector3D (0.00, 0.00, 0.00)>
   Nnext: <Vector3D (0.00, 0.00, 0.00)>

 Proximal Internal Coords:
   N: <Vector3D (-0.80, 1.27, -0.00)>
   Cα: <Vector3D (0.00, 0.00, 0.00)>
   C: <Vector3D (1.53, 0.00, 0.00)>
   O: <Vector3D (0.00, 0.00, 0.00)>
   Cβ: <Vector3D (-0.52, -0.90, -1.12)>
   Sγ: <Vector3D (-2.38, -0.96, -1.19)>
   Cprev <Vector3D (0.00, 0.00, 0.00

Now make a ``DisulfideList`` list and put the real structure and modelled structure into it.

In [8]:
minmax = DisulfideList([modelled_min, ssmin], "minmax")

Finally, display them in a common reference frame:

In [10]:
minmax.display_overlay(verbose=False)

Widget(value='<iframe src="http://localhost:59052/index.html?ui=P_0x16cc73fb0_2&reconnect=auto" class="pyvista…

The two structures overlap with an overall RMS error of 2.14 A. Not bad considering the modeled structure is made using idealized bond lengths and angles!

## References
* https://doi.org/10.1021/bi00368a023
